In [ ]:
import ee
ee.Authenticate()
ee.Initialize(project='gen-lang-client-0569405164')

assets = {
    "ottawa_da":  2045,
    "calgary_da": 1898,
    "quebec_da":  1317,
}

all_ok = True
for name, expected in assets.items():
    try:
        fc = ee.FeatureCollection(f'projects/gen-lang-client-0569405164/assets/{name}')
        n = fc.size().getInfo() # forces server-side count
        ok = (n == expected)
        all_ok = all_ok and ok
        print(f'{name:12s} features: {n:5d}  expected: {expected:5d}  {"OK" if ok else "MISMATCH — STOP"}')
    except Exception as e:
        all_ok = False
        print(f'{name:12s} ERROR: {e}')

print()
print("verified" if all_ok else "fix ingest")

ottawa_da    features:  2045  expected:  2045  OK
calgary_da   features:  1898  expected:  1898  OK
quebec_da    features:  1317  expected:  1317  OK

verified


ottawa_da    features:  2045  expected:  2045  OK
calgary_da   features:  1898  expected:  1898  OK
quebec_da    features:  1317  expected:  1317  OK

verified

---

all three exact, verified

export as 9.8 verbatim. three EE assets -> Daymet V4 warm season 2015-2019 -> reduce temp onto DA polygons server-side -> CSV to drive. same as 9.4

In [ ]:
import ee
ee.Initialize(project='gen-lang-client-0569405164')

city_boxes = {
    "ottawa_da":  ee.Geometry.Rectangle([-76.69, 44.80, -75.03, 46.04]),
    "calgary_da": ee.Geometry.Rectangle([-114.78, 50.74, -113.33, 51.54]),
    "quebec_da":  ee.Geometry.Rectangle([-71.86, 46.48, -70.66, 47.36]),
}
prefix = {"ottawa_da": "ottawa", "calgary_da": "calgary", "quebec_da": "quebec"}

for city, bbox in city_boxes.items():
    da_fc = ee.FeatureCollection(f'projects/gen-lang-client-0569405164/assets/{city}')
    daymet = (ee.ImageCollection('NASA/ORNL/DAYMET_V4')
              .filterDate('2015-05-01', '2019-10-01')
              .filterBounds(bbox).select(['tmax', 'tmin']))
    def keep_warm(img):
        m = ee.Date(img.get('system:time_start')).get('month')
        return img.set('keep', m.gte(5).And(m.lte(9)))
    warm = daymet.map(keep_warm).filter(ee.Filter.eq('keep', 1))
    def reduce_one_day(img):
        d = ee.Date(img.get('system:time_start')).format('YYYY-MM-dd')
        return img.reduceRegions(collection=da_fc, reducer=ee.Reducer.mean(),
                                 scale=1000, tileScale=4).map(lambda f: f.set('date', d))
    out = warm.map(reduce_one_day).flatten()
    ee.batch.Export.table.toDrive(
        collection=out, description=f'{prefix[city]}_daymet_2015_2019',
        folder='thesis/dlnm-pilot', fileNamePrefix=f'{prefix[city]}_daymet_2015_2019',
        fileFormat='CSV', selectors=['DAUID', 'date', 'tmax', 'tmin']).start()
    print(f'{city}: task started')

print('all 3 export tasks submitted')

ottawa_da: task started
calgary_da: task started
quebec_da: task started
all 3 export tasks submitted


ottawa_da: task started
calgary_da: task started
quebec_da: task started
all 3 export tasks submitted

---

all 3 export tasks submitted and running in EE task manager.

ottawa_da: task started
calgary_da: task started
quebec_da: task started
all 3 export tasks submitted

---

all three running on Earth Engine Task manager